<a href="https://colab.research.google.com/github/sofia-seo-j/chantey_2026/blob/BDS/BDS_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Cousre: Big Data Statistics*

S.E. Jeong (2894452)

# Assignment Part II

## Load

In [16]:
# Packages inmported
import pandas as pd
import statsmodels.api as sm
import numpy as np
from abess.linear import LinearRegression
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import math

In [17]:
# Data immported
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Assignment_BDS_25_26_data.csv')
print("\n Summary of the data:")
data.info()



 Summary of the data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540 entries, 0 to 539
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   radius_mean              540 non-null    float64
 1   texture_mean             540 non-null    float64
 2   perimeter_mean           540 non-null    float64
 3   area_mean                540 non-null    float64
 4   smoothness_mean          540 non-null    float64
 5   compactness_mean         540 non-null    float64
 6   concavity_mean           540 non-null    float64
 7   concave.points_mean      540 non-null    float64
 8   symmetry_mean            540 non-null    float64
 9   fractal_dimension_mean   540 non-null    float64
 10  radius_se                540 non-null    float64
 11  texture_se               540 non-null    float64
 12  perimeter_se             540 non-null    float64
 13  area_se                  540 non-null    float64
 14  smo

In [18]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


## Task 3

In [19]:
# Linear model with all the explanatory variables
X = data.drop(columns=["radius_mean"])
y = data["radius_mean"]

In [30]:
%%R -i X -i y
# install.packages("glmnet", repos="https://cloud.r-project.org")
library(glmnet)

X_mat <- as.matrix(X)
y_vec <- y

set.seed(1)
cv_model <- cv.glmnet(as.matrix(X), y, alpha=1, nfolds=10)

lambda_cv <- cv_model$lambda.min
cat("lambda (R):", lambda_cv, "\n")

coef_cv <- coef(cv_model, s = "lambda.min")

selected_vars <- rownames(coef_cv)[coef_cv[,1] != 0]
selected_vars <- selected_vars[selected_vars != "(Intercept)"]

cat("Non-zero coef. (R):", length(selected_vars), "\n")
print(selected_vars)

beta <- as.numeric(coef_cv)
beta0 <- beta[1]
beta_coefs <- beta[-1]

y_hat <- beta0 + X_mat %*% beta_coefs
residuals <- y_vec - y_hat

n <- length(y_vec)
lasso_obj <- (1/(2*n)) * sum(residuals^2) +
             lambda_cv * sum(abs(beta_coefs))

cat("LASSO criterion value:", lasso_obj, "\n")

lambda (R): 0.01741811 
Non-zero coef. (R): 6 
[1] "perimeter_mean"         "area_mean"              "compactness_mean"      
[4] "fractal_dimension_mean" "compactness_se"         "radius_worst"          
LASSO criterion value: 0.2651844 


In [31]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lasso = LassoCV(cv=10, random_state=1)
lasso.fit(X_scaled, y)

print("lambda:", lasso.alpha_)

selected_py = X.columns[lasso.coef_ != 0]
print("Non-zero coef.:", len(selected_py))
print(selected_py)

alpha = lasso.alpha_
coef = lasso.coef_
intercept = lasso.intercept_

# predictions
y_hat = lasso.predict(X_scaled)

# residuals
res = y - y_hat
n = len(y)

# LASSO criterion
lasso_obj = (1/(2*n)) * np.sum(res**2) + alpha * np.sum(np.abs(coef))

print("LASSO criterion:", lasso_obj)

lambda: 0.0034997041698912315
Non-zero coef.: 10
Index(['perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean',
       'concavity_mean', 'perimeter_se', 'concavity_se', 'concave.points_se',
       'radius_worst', 'texture_worst'],
      dtype='object')
LASSO criterion: 0.01717650688951551


## Task 4
To get a feel for the computational difficulties involved in best subset selection, assume that we have 100 explanatory variables and form subsets of size 25.

In [25]:
result = math.comb(100, 25)

result_str = str(result)
exponent = len(result_str) - 1
base_str = result_str[0] + '.' + result_str[1:]

print(f"{base_str} 10^{exponent}")
print(f"Exponent: {exponent}")

2.42519269720337121015504 10^23
Exponent: 23


In [23]:
# Here you can choose sequence as the option with support.size the sequence from 1 to 29.
model = LinearRegression(
    support_size=range(1, 30),
    ic_type='bic'
)

model.fit(X, y)

# Extract results
selected_idx = np.where(model.coef_ != 0)[0]
selected_vars = X.columns[selected_idx]

print("Selected support size:", model.support_size)
print("Number of selected variables:", len(selected_vars))
print("Selected variables:")
print(selected_vars)

Selected support size: range(1, 30)
Number of selected variables: 23
Selected variables:
Index(['texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean',
       'compactness_mean', 'concavity_mean', 'symmetry_mean',
       'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se',
       'area_se', 'smoothness_se', 'concavity_se', 'concave.points_se',
       'fractal_dimension_se', 'radius_worst', 'perimeter_worst', 'area_worst',
       'smoothness_worst', 'compactness_worst', 'concave.points_worst',
       'symmetry_worst'],
      dtype='object')
